In [ ]:
pip install folium


In [ ]:
pip install googlemaps pandas

  Preparing metadata (setup.py) ... done
  Created wheel for googlemaps: filename=googlemaps-4.10.0-py3-none-any.whl size=40714 sha256=29188926e621325ca100f0193fa1d4af9038fed18ebd7658572c59101293eb12
  Stored in directory: /root/.cache/pip/wheels/4c/6a/a7/bbc6f5c200032025ee655deb5e163ce8594fa05e67d973aad6
Successfully built googlemaps


In [ ]:
import pandas as pd
import googlemaps
from tqdm import tqdm
import random

# --- Add api key from google maps---
API_KEY = ''
INPUT_FILE = 'CrimeData.csv'
OUTPUT_FILE = 'CrimeData_Kandy_Strict.csv'
ADDRESS_COLUMN = 'location'

# --- 1. RANDOMIZATION SETTINGS ---
# This keeps points close to their real location but separates them visually.
SPREAD_FACTOR = 0.002

# --- 2. KANDY GEOGRAPHIC BOUNDARIES (Approximate) ---
KANDY_BOUNDS = {
    'min_lat': 6.9000,
    'max_lat': 7.5000,
    'min_lng': 80.3800,
    'max_lng': 81.0000
}

gmaps = googlemaps.Client(key=API_KEY)

# Load Data
print("Loading data...")
try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print(f"Error: {INPUT_FILE} not found.")
    exit()

unique_locations = df[ADDRESS_COLUMN].unique()
print(f"Found {len(unique_locations)} unique locations.")

# --- STRICT SEARCH FUNCTION ---
def get_strict_kandy_location(address_name):
    try:
        geocode_result = gmaps.geocode(
            address_name,
            components={'administrative_area': 'Kandy', 'country': 'LK'}
        )

        if not geocode_result:
            return None, None

        result = geocode_result[0]

        # LAYER 2: Text Verification
        address_text = str(result.get('formatted_address', '')).lower()
        is_valid_area = 'kandy' in address_text or 'central province' in address_text

        if not is_valid_area:
            for comp in result.get('address_components', []):
                if 'Kandy' in comp['long_name'] or 'Central Province' in comp['long_name']:
                    is_valid_area = True
                    break

        if is_valid_area:
            loc = result['geometry']['location']
            return loc['lat'], loc['lng']
        else:
            print(f"Skipped '{address_name}': Result was not in Kandy ({address_text})")
            return None, None

    except Exception as e:
        print(f"Error finding '{address_name}': {e}")
        return None, None

# Run Geocoding
location_map = {}
print("Starting STRICT geocoding...")

for loc in tqdm(unique_locations):
    if pd.isna(loc) or str(loc).strip() == "":
        location_map[loc] = (None, None)
        continue

    lat, lng = get_strict_kandy_location(loc)
    location_map[loc] = (lat, lng)

# --- APPLY JITTER  ---
print("Applying spatial jitter (staying within Kandy bounds)...")

def apply_strict_jitter(location_name):
    base_coords = location_map.get(location_name)
    if not base_coords or base_coords[0] is None:
        return None, None

    base_lat, base_lng = base_coords

    for _ in range(10):
        lat_offset = random.uniform(-SPREAD_FACTOR, SPREAD_FACTOR)
        lng_offset = random.uniform(-SPREAD_FACTOR, SPREAD_FACTOR)

        new_lat = base_lat + lat_offset
        new_lng = base_lng + lng_offset

        # LAYER 3: Bounding Box Check
        if (KANDY_BOUNDS['min_lat'] <= new_lat <= KANDY_BOUNDS['max_lat'] and
            KANDY_BOUNDS['min_lng'] <= new_lng <= KANDY_BOUNDS['max_lng']):
            return new_lat, new_lng

    return base_lat, base_lng

df['random_coords'] = df[ADDRESS_COLUMN].apply(apply_strict_jitter)

# Split and Save
df['Latitude'] = df['random_coords'].apply(lambda x: x[0] if x else None)
df['Longitude'] = df['random_coords'].apply(lambda x: x[1] if x else None)

df_final = df.drop(columns=['random_coords'])
df_final = df_final.dropna(subset=['Latitude', 'Longitude'])

df_final.to_csv(OUTPUT_FILE, index=False)
print(f"Success! Saved {len(df_final)} strict Kandy records to {OUTPUT_FILE}")

Loading data...
Found 147 unique locations.
Starting STRICT geocoding...


 35%|███▌      | 52/147 [00:00<00:01, 84.01it/s]

Skipped 'inside kurunegala bus': Result was not in Kandy (kurunegala, sri lanka)


 79%|███████▉  | 116/147 [00:01<00:00, 75.32it/s]

Skipped 'kandy car park': Result was not in Kandy (302 high level rd, colombo 00600, sri lanka)


100%|██████████| 147/147 [00:02<00:00, 62.86it/s]

Applying spatial jitter (staying within Kandy bounds)...
Success! Saved 1605 strict Kandy records to CrimeData_Kandy_Strict.csv


In [ ]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

INPUT_FILE = 'CrimeData_Kandy_Strict.csv'
OUTPUT_MAP = 'Crime_Map_Visualization.html'

# 1. Load the data
print("Loading data for visualization...")
try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print(f"Error: Could not find {INPUT_FILE}")
    exit()

# 2. Create a map
print("Creating map...")
m = folium.Map(location=[7.2906, 80.6337], zoom_start=13)

# 3. Add points to the map
marker_cluster = MarkerCluster().add_to(m)

count = 0
for index, row in df.iterrows():
    if pd.isna(row['Latitude']) or pd.isna(row['Longitude']):
        continue

    # Create a small circle for each crime record
    popup_text = f"Location: {row['location']}"

    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=5,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.6,
        popup=popup_text
    ).add_to(marker_cluster)

    count += 1

# 4. Save the map
m.save(OUTPUT_MAP)
print(f"Done! Added {count} points to the map.")
print(f"Open '{OUTPUT_MAP}' in your web browser to see the result.")

Loading data for visualization...
Creating map...
Done! Added 1605 points to the map.
Open 'Crime_Map_Visualization.html' in your web browser to see the result.
